In [7]:
import pandas as pd
import numpy as np
import json
import ast

In [8]:
movies=pd.read_csv('/content/drive/MyDrive/Archivos .csv/movies_metadata.csv')
creditos=pd.read_csv('/content/drive/MyDrive/Archivos .csv/credits.csv')

/tmp/ipython-input-269344844.py:1: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  movies=pd.read_csv('/content/drive/MyDrive/Archivos .csv/movies_metadata.csv')


In [9]:
# Limpieza de IDs para poder hacer el merge
# Convertimos a numérico, lo que sea error (basura) se convierte en NaN y lo borramos
movies['id'] = pd.to_numeric(movies['id'], errors='coerce')
movies = movies.dropna(subset=['id'])
movies['id'] = movies['id'].astype(int)

# 2. Convertir budget a numérico
movies['budget'] = pd.to_numeric(movies['budget'], errors='coerce')
movies['budget'] = movies['budget'].fillna(0)

# 3. Preparar creditos para el merge
creditos['id'] = creditos['id'].astype(int)

# 4. Fusión de los datasets
df = pd.merge(movies, creditos, on='id')

print(f"Dimensiones del dataset unido: {df.shape}")

Dimensiones del dataset unido: (45538, 26)


In [10]:
onehot=pd.DataFrame()

# Función para extraer el genero
def get_generos(objeto):
    lgeneros= map(lambda genero:genero['name'],objeto) #recorre la lista de diccionarios
    return ','.join(lgeneros) #concatena con comas los nombres extraidos

# Función para extraer el director
def get_director(x):
    try:
        if isinstance(x, str):
            x = ast.literal_eval(x)
        for i in x:
            if i['job'] == 'Director':
                return i['name']
        return np.nan
    except:
        return np.nan

# Función para extraer los 3 primeros actores (o devolver lista vacía)
def get_cast_names(x):
    try:
        if isinstance(x, str):
            x = ast.literal_eval(x)
        if isinstance(x, list):
            return [i['name'] for i in x][:3] # Top 3 actores
        return []
    except:
        return []

# Aplicamos las transformaciones
df['director'] = df['crew'].apply(get_director)
df['cast_top3'] = df['cast'].apply(get_cast_names)

# Para géneros (aprovechando tu lógica)
df['genres'] = df['genres'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df['genres_list'] = df['genres'].apply(lambda x: [i['name'] for i in x] if isinstance(x, list) else [])

In [11]:
# Cálculos de las constantes
m = df['vote_average'].mean() # Media de todo el dataset [cite: 15]
C = df['vote_count'].median() # Mediana de votos como confianza [cite: 18]

# Función para calcular la media bayesiana fila por fila
def weighted_rating(x, m=m, C=C):
    v = x['vote_count']
    R = x['vote_average']
    return (v / (v + C) * R) + (C / (v + C) * m)

# Creamos la columna objetivo (Target)
df['bayesian_rating'] = df.apply(weighted_rating, axis=1)

In [12]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
genres_encoded = pd.DataFrame(mlb.fit_transform(df['genres_list']), columns=mlb.classes_, index=df.index)

# Unimos con el dataframe principal
df_model = pd.concat([df[['budget', 'bayesian_rating']], genres_encoded], axis=1)

# Eliminamos filas donde budget es 0 (opcional, pero recomendado para mejorar calidad)
df_model = df_model[df_model['budget'] > 1000]

In [14]:
from sklearn.model_selection import StratifiedShuffleSplit
import pandas as pd
import numpy as np

# ASUMIENDO: df_model YA está creado con la Media Bayesiana y los géneros One-Hot,
# y ya se ha aplicado el filtrado de presupuesto.

# === FIX APLICADO AQUÍ ===
# Reiniciar el índice para asegurar que sea contiguo (0, 1, 2, 3, ...)
df_model = df_model.reset_index(drop=True)
# =========================

# 1. Crear categorías de presupuesto para la estratificación
# Dividimos el budget en 5 grupos (bins)
df_model['budget_cat'] = pd.cut(df_model['budget'],
                               bins=[0, 1e6, 10e6, 50e6, 100e6, np.inf],
                               labels=[1, 2, 3, 4, 5])

# 2. Inicializar el Splitter
split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

# 3. Generar los índices
for train_index, test_index in split.split(df_model, df_model['budget_cat']):
    strat_train_set = df_model.loc[train_index]
    strat_test_set = df_model.loc[test_index]

# 4. Eliminar la categoría auxiliar para no usarla en el entrenamiento
for set_ in (strat_train_set, strat_test_set):
    set_.drop("budget_cat", axis=1, inplace=True)

# Separar X e y
X_train = strat_train_set.drop("bayesian_rating", axis=1)
y_train = strat_train_set["bayesian_rating"]
X_test = strat_test_set.drop("bayesian_rating", axis=1)
y_test = strat_test_set["bayesian_rating"]

In [15]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# --- Regresión Lineal ---
pipeline_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('lin_reg', LinearRegression())
])

pipeline_lr.fit(X_train, y_train)
predictions_lr = pipeline_lr.predict(X_test)

print("Linear Regression RMSE:", np.sqrt(mean_squared_error(y_test, predictions_lr)))
print("Linear Regression R2:", r2_score(y_test, predictions_lr))

# --- Random Forest ---
rf_reg = RandomForestRegressor(n_estimators=100, random_state=42)
rf_reg.fit(X_train, y_train)
predictions_rf = rf_reg.predict(X_test)

print("Random Forest RMSE:", np.sqrt(mean_squared_error(y_test, predictions_rf)))
print("Random Forest R2:", r2_score(y_test, predictions_rf))

Linear Regression RMSE: 0.7474888496624893
Linear Regression R2: 0.1491805091469629
Random Forest RMSE: 0.7991733809893722
Random Forest R2: 0.02745431163971168
